# Portfolio Risk Management: Factor Models, Stress Testing and Crisis Hedging

This notebook covers the full workflow of:
* factor-model-driven portfolio construction
* stress testing under two geopolitical crises
* risk-based rebalancing

**Two crisis tracks:**
- Gulf Crisis: oil spike, stagflation, EM stress
- Ukraine/Russia: energy and food shock, defense surge, European risk premium

**Workflows:**
1. BUILD: Build factor model and benign portfolio
2. BUILD: Define crisis portfolios for each scenario
3. RISK: Stress test benign portfolio against both crises
4. RISK: Hedge and rebalance
5. ANALYSIS: Compare outcomes

---
## Part 1: Factor Model Foundations

### 1.1 Basics

A factor model decomposes portfolio returns into:
- **Systematic returns**: driven by common risk factors
- **Idiosyncratic returns**: asset-specific, diversifiable

$$
r_i = \alpha_i + \sum_{k=1}^{K} \beta_{ik} f_k + \epsilon_i
$$

where:
- $r_i$ = return of asset $i$
- $\beta_{ik}$ = exposure (loading) of asset $i$ to factor $k$
- $f_k$ = return of factor $k$
- $\epsilon_i$ = idiosyncratic return, $\epsilon_i \sim \mathcal{N}(0, \sigma_i^2)$

Portfolio return:
$$
r_p = \sum_i w_i r_i = \sum_{k} \left(\sum_i w_i \beta_{ik}\right) f_k + \sum_i w_i \epsilon_i = \sum_k B_{pk} f_k + \epsilon_p
$$

Portfolio variance:
$$
\sigma_p^2 = \mathbf{B}_p^\top \mathbf{F} \mathbf{B}_p + \mathbf{w}^\top \mathbf{\Delta} \mathbf{w}
$$

where $\mathbf{F}$ is the factor covariance matrix and $\mathbf{\Delta}$ is the diagonal idiosyncratic variance matrix.

### 1.2 Factor Selection

For a multi-asset portfolio spanning equities, rates, and geopolitical risks, we use three layers of factors:

**Macro factors** — drive cross-asset returns

| Factor | Proxy | Rationale |
|---|---|---|
| Oil price | Brent crude return | Key driver of inflation, EM, energy sector |
| Inflation surprise | 5Y breakeven rate change | Reprices bonds and rate-sensitive equities |
| Real rates | 5Y TIPS yield change | Core driver of bond and growth equity valuations |
| USD strength | DXY return | EM stress, commodity prices, risk-off |
| Credit spread | IG/HY OAS change | Risk appetite, funding conditions |
| Geopolitical risk | GPR index change | Tail risk, defense, safe havens |
| Food/agriculture | Bloomberg Agri index | Ukraine-specific transmission |

where: 
* **DXY**: US Dollar Index acroos basket of major currencies (EUR, JPY, GBP, CAD, SEK, CHF)
* **OAS**: Option Adjusted Spread
* **GPR**: Geopol risk index: uses newspaper text analysis

**Style factors** — drive cross-sectional equity returns

| Factor | Definition | Rationale |
|---|---|---|
| Momentum | 12M-1M price return | Trend following, crisis amplification |
| Value | B/P ratio | Mean reversion, cheap vs expensive |
| Quality | ROE, low leverage | Defensive in stress |
| Size | Market cap | Small cap more vulnerable in risk-off |
| Low volatility | Realized vol | Defensive factor |

**Sector/regional factors** — capture industry and geography

| Factor | Relevance |
|---|---|
| Energy | Gulf and Ukraine direct exposure |
| Defense | Ukraine specific |
| Tourism/hospitality | Gulf specific |
| EM Asia | Growth factor, oil importer stress |
| European equities | Ukraine proximity premium |
| Insurance | Rate sensitivity, cat risk |

### 1.3 Factor Covariance Matrix

In practice, $\mathbf{F}$ is estimated from historical returns using a shrinkage estimator to avoid overfitting:

$$
\hat{\mathbf{F}} = (1 - \delta) \mathbf{S} + \delta \mathbf{T}
$$

where $\mathbf{S}$ is the sample covariance, $\mathbf{T}$ is the shrinkage target (e.g. constant correlation), and $\delta$ is the shrinkage intensity estimated via Ledoit-Wolf.

Factor exposures $\beta_{ik}$ are estimated via time-series OLS for macro factors, and cross-sectional regression for style factors.

In [ ]:
from quant_risk.setup import base, macro

np, pd, plt = base()
fed_client, store, external_store = macro()

In [ ]:
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.covariance import LedoitWolf
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

In [ ]:
fred_series = [
    "US10Y",
    "US2Y",
    "VIX",
    "IG_OAS",
    "HY_OAS",
    "BRENT",
    "BREAKEVEN5Y",
    "REAL5Y",
    "EURUSD",
    "DXY",
]

df_fred = store.build_panel(fred_series)
print(df_fred.shape)
print(df_fred.columns.tolist())
print(df_fred.isna().sum())

In [ ]:
ff = external_store.get_fama_french(frequency='daily')

style_factors = pd.DataFrame({
    'momentum': ff['Mom'],
    'value':    ff['HML'],
    'quality':  ff['RMW'],
    'size':     ff['SMB'],
    'low_vol':  -ff['Mkt-RF'],
}).dropna()

print(style_factors.head())
print(style_factors.shape)

In [ ]:
yf_series = [
    "ENERGY_SECTOR",
    "DEFENSE_SECTOR",
    "EUROPE_EQ",
    "EM_ASIA",
    "INSURANCE",
    "TOURISM",
    "AGRICULTURE",
    "GLD",
    "TLT",
    "EEM",
    "IEF",
    "LQD",
    "HYG",
    "EWG",
    "EWI",
]

df_yf = external_store.build_panel(yf_series)
print(df_yf.shape)
print(df_yf.columns.tolist())
print(df_yf.isna().sum())

In [ ]:
# GPR
gpr = external_store.get_gpr()
gpr.head()

In [ ]:
import yfinance as yf

tickers = {
    "ENERGY_SECTOR":  "XLE",
    "DEFENSE_SECTOR": "ITA",
    "EUROPE_EQ":      "FEZ",
    "EM_ASIA":        "GMF",
    "INSURANCE":      "IAK",
    "TOURISM":        "PEJ",
    "AGRICULTURE":    "^SPGSAG",
}

for name, ticker in tickers.items():
    raw = yf.download(ticker, start="2000-01-01", auto_adjust=True, progress=False)
    close = raw["Close"]
    print(f"{name}: type={type(close).__name__}, shape={close.shape}, columns={close.columns.tolist() if hasattr(close, 'columns') else 'n/a'}")

In [ ]:
# ── Simulate factor return history (250 trading days) ───────────────────────
# In production these come from a vendor or in-house estimation pipeline.
# Annualised vols and cross-factor correlations are calibrated to realistic levels.

T = 250  # trading days

# Annualised vols per factor (rough empirical calibration)
factor_vols = np.array([
    0.35,  # oil
    0.015, # inflation surprise
    0.012, # real rates
    0.07,  # usd
    0.30,  # credit spread
    0.25,  # geo risk
    0.20,  # agriculture
    0.15,  # momentum
    0.12,  # value
    0.10,  # quality
    0.13,  # size
    0.10,  # low vol
    0.30,  # energy sector
    0.25,  # defense sector
    0.25,  # tourism sector
    0.22,  # em asia
    0.20,  # europe eq
    0.22,  # insurance sector
])

daily_vols = factor_vols / np.sqrt(252)

# Correlation structure: block structure by factor group
rho = np.eye(N_FACTORS)

# Macro block: oil, inflation, real rates, usd, credit, geo, agri
macro_corr = [
    [1.00,  0.45,  0.20, -0.30,  0.15,  0.30,  0.25],
    [0.45,  1.00,  0.35, -0.20,  0.10,  0.15,  0.20],
    [0.20,  0.35,  1.00, -0.10,  0.25,  0.05,  0.10],
    [-0.30,-0.20, -0.10,  1.00,  0.20,  0.25, -0.15],
    [0.15,  0.10,  0.25,  0.20,  1.00,  0.20,  0.10],
    [0.30,  0.15,  0.05,  0.25,  0.20,  1.00,  0.15],
    [0.25,  0.20,  0.10, -0.15,  0.10,  0.15,  1.00],
]
n_macro = len(MACRO_FACTORS)
rho[:n_macro, :n_macro] = macro_corr

# Cholesky for simulation
cov_matrix = np.outer(daily_vols, daily_vols) * rho
# Ensure PSD
eigvals = np.linalg.eigvalsh(cov_matrix)
if eigvals.min() < 0:
    cov_matrix += (-eigvals.min() + 1e-8) * np.eye(N_FACTORS)

factor_returns = np.random.multivariate_normal(
    mean=np.zeros(N_FACTORS),
    cov=cov_matrix,
    size=T
)  # shape (T, N_FACTORS)

factor_df = pd.DataFrame(factor_returns, columns=ALL_FACTORS)
print('Factor return history shape:', factor_df.shape)
print('\nAnnualised vols (simulated):')
print((factor_df.std() * np.sqrt(252)).round(3).to_string())

In [ ]:
# ── Ledoit-Wolf factor covariance matrix ─────────────────────────────────────
lw = LedoitWolf().fit(factor_df)
F_cov = pd.DataFrame(lw.covariance_, index=ALL_FACTORS, columns=ALL_FACTORS)

# Plot factor correlation matrix
F_corr = F_cov.copy()
vols    = np.sqrt(np.diag(F_cov.values))
F_corr  = F_cov / np.outer(vols, vols)

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(
    F_corr.round(2), annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1, ax=ax, annot_kws={'size': 7},
    linewidths=0.3
)
ax.set_title('Factor Correlation Matrix (Ledoit-Wolf)', fontsize=13, pad=12)
plt.tight_layout()
plt.show()

---
## Part 2: Benign Portfolio Construction

### 2.1 Macro Scenario: Low Inflation, Growth, EM, Rate Cuts

The benign scenario is defined by:
- Inflation anchored at 2%, central banks cutting rates
- Global growth above trend, EM outperforming DM
- Oil stable to declining, USD soft
- Credit spreads tight, risk appetite high

**Desired factor exposures:**

| Factor | Target exposure | Rationale |
|---|---|---|
| Oil | Neutral to slight negative | Low commodity price thesis |
| Inflation surprise | Negative | Long duration, hurt by inflation |
| Real rates | Negative | Long bonds, hurt by rising real rates |
| USD | Negative | Long EM, soft USD benefits |
| Credit spread | Negative | Risk-on, tight spreads |
| Geo risk | Negative | No geopolitical premium priced in |
| Momentum | Positive | Trend following in growth assets |
| Quality | Neutral | Not a defensive positioning |
| EM Asia | Positive | Core thesis |
| Tourism | Positive | Recovery play |
| Insurance | Positive | Rate beneficiary |
| Defense | Zero | No crisis scenario |
| Energy sector | Neutral | Low oil, no energy overweight |

### 2.2 Portfolio Legs and Asset-Factor Loadings

Each portfolio leg maps onto the factor model via its $\beta$ vector. Below are the assumed loadings based on the asset class characteristics.

In [ ]:
# ── Asset universe and factor loadings ───────────────────────────────────────
# Rows = assets, Columns = factors
# Loadings calibrated to reflect economic intuition for each asset class.

ASSETS = [
    '5Y_bonds',
    '30Y_bonds',
    'tourism',
    'em_asia_eq',
    'insurance',
    'sp500_put',   # put = negative beta to equities, positive to vol
    'cash',
]

# Beta matrix: shape (n_assets, n_factors)
# Convention: positive beta = moves with factor
# Real rates and inflation: positive = hurt when factor rises (bond-like)
betas = pd.DataFrame({
    # Macro
    'oil':            [ 0.00,  0.00,  -0.30, -0.25,  -0.10,   0.10,  0.00],
    'inflation_surp': [-0.40, -1.20,  -0.10, -0.05,  -0.15,   0.05,  0.00],
    'real_rates':     [-0.35, -1.50,  -0.15, -0.20,  -0.20,   0.05,  0.00],
    'usd':            [ 0.00,  0.00,  -0.10, -0.45,  -0.05,   0.10,  0.00],
    'credit_spread':  [-0.05, -0.10,  -0.20, -0.30,  -0.25,   0.15,  0.00],
    'geo_risk':       [-0.05, -0.10,  -0.35, -0.30,  -0.20,   0.20,  0.00],
    'agriculture':    [ 0.00,  0.00,  -0.05, -0.10,  -0.05,   0.02,  0.00],
    # Style
    'momentum':       [ 0.05,  0.05,   0.20,  0.25,   0.15,  -0.30,  0.00],
    'value':          [ 0.00,  0.00,   0.10,  0.15,   0.20,  -0.10,  0.00],
    'quality':        [ 0.10,  0.10,   0.05,  0.05,   0.15,  -0.05,  0.00],
    'size':           [ 0.00,  0.00,   0.15,  0.20,   0.10,  -0.15,  0.00],
    'low_vol':        [ 0.20,  0.25,  -0.10, -0.10,   0.05,   0.10,  0.00],
    # Sector
    'energy_sector':  [ 0.00,  0.00,  -0.10, -0.05,  -0.05,   0.05,  0.00],
    'defense_sector': [ 0.00,  0.00,  -0.05, -0.05,   0.00,   0.05,  0.00],
    'tourism_sector': [ 0.00,  0.00,   0.85,  0.10,   0.05,  -0.20,  0.00],
    'em_asia':        [ 0.00,  0.00,   0.10,  0.80,   0.05,  -0.20,  0.00],
    'europe_eq':      [ 0.00,  0.00,   0.10,  0.15,   0.20,  -0.15,  0.00],
    'insurance_sector':[ 0.00, 0.00,   0.05,  0.10,   0.75,  -0.15,  0.00],
}, index=ASSETS).T  # shape (n_factors, n_assets)

print('Beta matrix shape (factors x assets):', betas.shape)
betas.round(2)

In [ ]:
# ── Benign portfolio weights ─────────────────────────────────────────────────
# As discussed: % of NAV
w_benign = pd.Series({
    '5Y_bonds':   0.30,
    '30Y_bonds':  0.10,
    'tourism':    0.10,
    'em_asia_eq': 0.20,
    'insurance':  0.20,
    'sp500_put':  0.05,
    'cash':       0.05,
})

assert abs(w_benign.sum() - 1.0) < 1e-9, 'Weights must sum to 1'

# ── Portfolio factor exposures: B_p = betas @ w ──────────────────────────────
B_benign = betas.values @ w_benign.values  # shape (n_factors,)
B_benign = pd.Series(B_benign, index=ALL_FACTORS)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#d32f2f' if v < 0 else '#1565c0' for v in B_benign]
ax.barh(B_benign.index, B_benign.values, color=colors, edgecolor='white', height=0.6)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Benign Portfolio: Factor Exposures', fontsize=13)
ax.set_xlabel('Factor Beta (portfolio level)')
plt.tight_layout()
plt.show()

print('\nPortfolio factor exposures:')
print(B_benign.round(3).to_string())

In [ ]:
# ── Portfolio risk decomposition ─────────────────────────────────────────────
# Total portfolio variance = factor variance + idiosyncratic variance
# sigma_p^2 = B_p' F B_p + w' Delta w

# Idiosyncratic vols (annualised) per asset
idio_vols = pd.Series({
    '5Y_bonds':   0.03,
    '30Y_bonds':  0.06,
    'tourism':    0.18,
    'em_asia_eq': 0.15,
    'insurance':  0.14,
    'sp500_put':  0.30,
    'cash':       0.00,
})

Delta = np.diag((idio_vols.values / np.sqrt(252))**2)  # daily idio variance

# Factor contribution to variance
B_p     = betas.values @ w_benign.values  # (n_factors,)
F_daily = F_cov.values                    # already daily from simulation
factor_var  = B_p @ F_daily @ B_p
idio_var    = w_benign.values @ Delta @ w_benign.values
total_var   = factor_var + idio_var

# Annualised vols
total_vol   = np.sqrt(total_var * 252)
factor_vol  = np.sqrt(factor_var * 252)
idio_vol_p  = np.sqrt(idio_var * 252)

print(f'Portfolio annualised volatility: {total_vol:.2%}')
print(f'  Factor component:              {factor_vol:.2%}  ({factor_var/total_var:.0%} of total variance)')
print(f'  Idiosyncratic component:       {idio_vol_p:.2%}  ({idio_var/total_var:.0%} of total variance)')

# Factor-by-factor variance contribution
# Marginal contribution of factor k: 2 * B_pk * (F B_p)_k  (not marginal, full contribution)
F_Bp            = F_daily @ B_p
factor_contrib  = B_p * F_Bp  # element-wise, shape (n_factors,)
factor_contrib  = pd.Series(factor_contrib, index=ALL_FACTORS)
factor_contrib_pct = factor_contrib / total_var

fig, ax = plt.subplots(figsize=(10, 5))
fc_sorted = factor_contrib_pct.sort_values()
colors = ['#d32f2f' if v < 0 else '#1565c0' for v in fc_sorted]
ax.barh(fc_sorted.index, fc_sorted.values * 100, color=colors, edgecolor='white', height=0.6)
ax.axvline(0, color='black', linewidth=0.8)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title('Benign Portfolio: Factor Contribution to Total Variance', fontsize=13)
ax.set_xlabel('% of total variance')
plt.tight_layout()
plt.show()

---
## Part 3a: Gulf Crisis — Factor Shock Definition and Crisis Portfolio

### 3a.1 Scenario Narrative

The Gulf crisis archetype (1990 Gulf War, hypothetical 2020s escalation) transmits through markets via:

- **Oil spike**: supply disruption, Brent +30% to +40% in weeks
- **Inflation surge**: oil feeds headline CPI; breakevens reprice sharply
- **Rate repricing**: central banks cannot cut; long yields rise 80-120bp
- **USD strength**: risk-off, safe haven demand
- **EM stress**: oil importers squeezed, capital outflows, FX weakness
- **Equity dispersion**: energy and defense surge; tourism, EM, insurance bleed
- **S&P index resilient**: energy weight offsets losers, index flat to slightly down

### 3a.2 Factor Shocks (Gulf Crisis)

| Factor | Shock | Direction | Rationale |
|---|---|---|---|
| Oil | +35% | Up | Supply disruption |
| Inflation surprise | +120bp | Up | Oil pass-through to CPI |
| Real rates | +80bp | Up | Yields rise, no CB support |
| USD | +6% | Up | Safe haven |
| Credit spread | +80bp | Up | Risk-off |
| Geo risk | +40% | Up | Conflict premium |
| Agriculture | +5% | Up | Minor, not Gulf-specific |
| Momentum | -15% | Down | Trend breaks in crisis |
| Tourism sector | -25% | Down | Travel disruption, fuel costs |
| EM Asia | -20% | Down | Oil importer stress |
| Insurance sector | -12% | Down | Bond MTM losses dominate |
| Defense sector | +20% | Up | Conflict premium |
| Energy sector | +30% | Up | Direct beneficiary |

In [ ]:
# ── Gulf crisis factor shocks ────────────────────────────────────────────────
# Expressed as total return shocks over the crisis horizon (weeks to months)

gulf_shocks = pd.Series({
    'oil':             0.35,
    'inflation_surp':  0.012,   # 120bp in rate terms
    'real_rates':      0.008,   # 80bp
    'usd':             0.06,
    'credit_spread':   0.008,   # 80bp
    'geo_risk':        0.40,
    'agriculture':     0.05,
    'momentum':       -0.15,
    'value':           0.05,
    'quality':         0.08,
    'size':           -0.10,
    'low_vol':         0.10,
    'energy_sector':   0.30,
    'defense_sector':  0.20,
    'tourism_sector': -0.25,
    'em_asia':        -0.20,
    'europe_eq':      -0.08,
    'insurance_sector':-0.12,
}, name='Gulf Crisis')

# ── Ukraine/Russia factor shocks ─────────────────────────────────────────────
ukraine_shocks = pd.Series({
    'oil':             0.25,
    'inflation_surp':  0.015,   # 150bp: gas + food compound
    'real_rates':      0.006,
    'usd':             0.05,
    'credit_spread':   0.006,
    'geo_risk':        0.60,    # larger geopolitical shock
    'agriculture':     0.35,    # wheat, corn, sunflower: Ukraine is major exporter
    'momentum':       -0.10,
    'value':           0.08,
    'quality':         0.10,
    'size':           -0.08,
    'low_vol':         0.12,
    'energy_sector':   0.25,
    'defense_sector':  0.40,    # much stronger defense surge
    'tourism_sector': -0.15,
    'em_asia':        -0.12,
    'europe_eq':      -0.20,    # proximity hit, energy dependency
    'insurance_sector':-0.10,
}, name='Ukraine/Russia')

shocks_df = pd.DataFrame({'Gulf': gulf_shocks, 'Ukraine': ukraine_shocks})

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (col, color) in zip(axes, [('Gulf', '#b71c1c'), ('Ukraine', '#1a237e')]):
    vals = shocks_df[col].sort_values()
    colors = [color if v > 0 else '#555555' for v in vals]
    ax.barh(vals.index, vals.values * 100, color=colors, edgecolor='white', height=0.6)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_title(f'{col} Crisis: Factor Shocks', fontsize=12)
plt.tight_layout()
plt.show()

---
## Part 4: Stress Test — Benign Portfolio vs Both Crises

### 4.1 P&L Attribution

For a given factor shock vector $\Delta f$, the portfolio P&L is:

$$
\text{P\&L}_p = \sum_{k} B_{pk} \cdot \Delta f_k
$$

The contribution of each asset leg $i$:

$$
\text{P\&L}_i = w_i \sum_{k} \beta_{ik} \cdot \Delta f_k
$$

This decomposes total loss into factor and asset contributions simultaneously.

In [ ]:
def stress_test(weights: pd.Series, betas: pd.DataFrame, shocks: pd.Series) -> dict:
    """
    Run a factor-model stress test.

    Parameters
    ----------
    weights : pd.Series  (assets,)
    betas   : pd.DataFrame  (factors x assets)
    shocks  : pd.Series  (factors,)  -- total return shocks

    Returns
    -------
    dict with total P&L, factor attribution, asset attribution
    """
    # Asset-level P&L = sum_k beta_ik * shock_k, then weighted
    asset_pnl = betas.T @ shocks  # (assets,)
    weighted_pnl = weights * asset_pnl  # (assets,), P&L contribution
    total_pnl = weighted_pnl.sum()

    # Factor-level attribution
    B_p = betas @ weights  # (factors,)
    factor_pnl = B_p * shocks  # (factors,)

    return {
        'total_pnl':    total_pnl,
        'asset_pnl':    weighted_pnl,
        'factor_pnl':   factor_pnl,
        'asset_return': asset_pnl,
    }


# Run stress tests on benign portfolio
gulf_stress    = stress_test(w_benign, betas, gulf_shocks)
ukraine_stress = stress_test(w_benign, betas, ukraine_shocks)

print('=== BENIGN PORTFOLIO STRESS TEST ===')
print(f"\nGulf Crisis total P&L:          {gulf_stress['total_pnl']:+.2%} of NAV")
print(f"Ukraine/Russia total P&L:       {ukraine_stress['total_pnl']:+.2%} of NAV")

print('\n--- Gulf: P&L by asset ---')
print(gulf_stress['asset_pnl'].sort_values().apply(lambda x: f'{x:+.2%}').to_string())

print('\n--- Ukraine: P&L by asset ---')
print(ukraine_stress['asset_pnl'].sort_values().apply(lambda x: f'{x:+.2%}').to_string())

In [ ]:
# ── P&L attribution plots ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

scenarios = [
    ('Gulf Crisis', gulf_stress, '#b71c1c'),
    ('Ukraine/Russia', ukraine_stress, '#1a237e'),
]

for col, (name, stress, base_color) in enumerate(scenarios):
    # Asset attribution
    ax = axes[0][col]
    vals = stress['asset_pnl'].sort_values()
    colors = [base_color if v > 0 else '#888888' for v in vals]
    ax.barh(vals.index, vals.values * 100, color=colors, edgecolor='white', height=0.6)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_title(f'{name}: P&L by Asset (% NAV)', fontsize=11)
    ax.axvline(stress['total_pnl'] * 100, color=base_color, linewidth=1.5,
               linestyle='--', label=f"Total: {stress['total_pnl']:+.1%}")
    ax.legend(fontsize=9)

    # Factor attribution
    ax = axes[1][col]
    fvals = stress['factor_pnl'].sort_values()
    fcolors = [base_color if v > 0 else '#888888' for v in fvals]
    ax.barh(fvals.index, fvals.values * 100, color=fcolors, edgecolor='white', height=0.6)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_title(f'{name}: P&L by Factor (% NAV)', fontsize=11)

plt.suptitle('Benign Portfolio: Stress Test Attribution', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## Part 5: Hedging and Rebalancing

### 5.1 Diagnostic: What is Killing the Portfolio

From the stress test, the primary loss drivers in both crises are:

**Gulf:**
- Real rates and inflation surprise hitting the bond book
- Tourism sector direct hit
- EM Asia oil importer stress
- Insurance bond MTM losses

**Ukraine:**
- Same bond book losses
- European equity exposure
- Agriculture shock (secondary)
- Defense is a missing long

### 5.2 Hedging Instruments

The hedge targets the **factor exposures** that are causing losses, not individual positions.

| Factor exposure to hedge | Instrument | Mechanics |
|---|---|---|
| Real rates (long) | Short bond futures (Bund, Treasury) | Duration hedge |
| Inflation surprise | Long inflation breakevens / TIPS | Pays when inflation rises |
| Oil (indirect) | Long oil futures or energy ETF | Pays when oil spikes |
| USD (short EM) | Long USD / short EM FX via puts | Protects EM leg |
| Geo risk | Long VIX calls or gold | Crisis hedge |
| Defense (missing) | Long defense ETF | Ukraine-specific |

### 5.3 Hedge Sizing

For a target factor exposure $B_{pk}^{\text{target}}$, the hedge notional $h_k$ in instrument $j$ with loading $\beta_{jk}$ is:

$$
h_j = \frac{B_{pk}^{\text{current}} - B_{pk}^{\text{target}}}{\beta_{jk}}
$$

Target: reduce the most damaging factor exposures to zero or near zero.

In [ ]:
# ── Hedge instruments and their factor loadings ──────────────────────────────
HEDGE_INSTRUMENTS = [
    'bond_fut_short',  # short bond futures: negative real_rates and inflation beta
    'tips_long',       # long TIPS / breakevens
    'oil_fut_long',    # long oil futures
    'usd_call',        # long USD call / EM FX put
    'gold_long',       # gold: geo risk hedge
    'defense_etf',     # long defense: Ukraine-specific
]

hedge_betas = pd.DataFrame({
    'oil':             [-0.05,  0.00,  0.95,  0.10,  0.15,  0.05],
    'inflation_surp':  [ 0.80,  0.70,  0.20,  0.00,  0.10,  0.00],
    'real_rates':      [ 0.90,  0.30,  0.10, -0.05,  0.05,  0.00],
    'usd':             [ 0.00,  0.00,  0.10,  0.85, -0.10,  0.05],
    'credit_spread':   [ 0.10,  0.05,  0.05,  0.10,  0.05,  0.00],
    'geo_risk':        [-0.05,  0.00,  0.15,  0.10,  0.60,  0.20],
    'agriculture':     [ 0.00,  0.00,  0.10,  0.00,  0.05,  0.00],
    'momentum':        [-0.10, -0.05,  0.10,  0.05, -0.05, -0.05],
    'value':           [ 0.00,  0.00,  0.05,  0.00,  0.05,  0.10],
    'quality':         [ 0.10,  0.10,  0.00,  0.00,  0.10,  0.15],
    'size':            [ 0.00,  0.00,  0.05,  0.00,  0.00,  0.10],
    'low_vol':         [ 0.15,  0.10,  0.00,  0.00,  0.20,  0.05],
    'energy_sector':   [ 0.00,  0.00,  0.80,  0.05,  0.10,  0.05],
    'defense_sector':  [ 0.00,  0.00,  0.00,  0.00,  0.05,  0.85],
    'tourism_sector':  [ 0.00,  0.00,  0.00,  0.00,  0.00,  0.00],
    'em_asia':         [ 0.00,  0.00,  0.00, -0.40,  0.00,  0.00],
    'europe_eq':       [ 0.00,  0.00,  0.00,  0.05, -0.05,  0.10],
    'insurance_sector':[ 0.05,  0.05,  0.00,  0.00,  0.00,  0.00],
}, index=HEDGE_INSTRUMENTS).T

print('Hedge instruments defined:', HEDGE_INSTRUMENTS)

In [ ]:
def compute_hedged_portfolio(
    w_base: pd.Series,
    betas_base: pd.DataFrame,
    hedge_betas: pd.DataFrame,
    shocks: pd.Series,
    hedge_budget: float = 0.10,
    target_factors: list = None,
) -> tuple:
    """
    Find hedge weights that minimise stressed P&L on target factors,
    subject to a notional budget constraint.

    Parameters
    ----------
    w_base        : base portfolio weights
    betas_base    : factor loadings of base portfolio (factors x assets)
    hedge_betas   : factor loadings of hedge instruments (factors x instruments)
    shocks        : factor shock vector
    hedge_budget  : max total notional of hedges (% NAV)
    target_factors: factors to hedge (None = all)

    Returns
    -------
    hedge_weights, hedged_pnl_dict
    """
    if target_factors is None:
        target_factors = ALL_FACTORS

    # Base portfolio factor exposures
    B_p = betas_base @ w_base  # (n_factors,)

    # Objective: minimise total stressed P&L of hedged portfolio
    # P&L_hedged = sum_k (B_pk + sum_j h_j * beta_jk) * shock_k
    # Minimise over h_j

    n_hedges = len(HEDGE_INSTRUMENTS)
    shock_vec = shocks.values  # (n_factors,)

    def objective(h):
        B_hedge = hedge_betas.values @ h   # (n_factors,)
        B_total = B_p.values + B_hedge
        return (B_total * shock_vec).sum()

    constraints = [
        {'type': 'ineq', 'fun': lambda h: hedge_budget - np.abs(h).sum()},
    ]
    bounds = [(-hedge_budget, hedge_budget)] * n_hedges
    h0 = np.zeros(n_hedges)

    result = minimize(objective, h0, method='SLSQP',
                      bounds=bounds, constraints=constraints,
                      options={'ftol': 1e-9, 'maxiter': 1000})

    h_opt = pd.Series(result.x, index=HEDGE_INSTRUMENTS)

    # Compute hedged stress test
    # Combine base weights and hedge weights into augmented portfolio
    w_full   = pd.concat([w_base, h_opt])
    beta_full = pd.concat([betas_base, hedge_betas], axis=1)
    hedged_stress = stress_test(w_full, beta_full, shocks)

    return h_opt, hedged_stress


# Hedge for Gulf crisis
h_gulf, gulf_hedged = compute_hedged_portfolio(
    w_benign, betas, hedge_betas, gulf_shocks, hedge_budget=0.12
)

# Hedge for Ukraine crisis
h_ukraine, ukraine_hedged = compute_hedged_portfolio(
    w_benign, betas, hedge_betas, ukraine_shocks, hedge_budget=0.12
)

print('=== HEDGE WEIGHTS ===')
print('\nGulf hedge:')
print(h_gulf[h_gulf.abs() > 0.001].round(3).apply(lambda x: f'{x:+.2%}').to_string())
print('\nUkraine hedge:')
print(h_ukraine[h_ukraine.abs() > 0.001].round(3).apply(lambda x: f'{x:+.2%}').to_string())

print('\n=== P&L COMPARISON ===')
print(f"Gulf    - Unhedged: {gulf_stress['total_pnl']:+.2%}  |  Hedged: {gulf_hedged['total_pnl']:+.2%}")
print(f"Ukraine - Unhedged: {ukraine_stress['total_pnl']:+.2%}  |  Hedged: {ukraine_hedged['total_pnl']:+.2%}")

In [ ]:
# ── Hedge effectiveness plot ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, unhedged, hedged, color) in zip(axes, [
    ('Gulf Crisis',    gulf_stress,    gulf_hedged,    '#b71c1c'),
    ('Ukraine/Russia', ukraine_stress, ukraine_hedged, '#1a237e'),
]):
    # Compare unhedged vs hedged asset P&L
    # For hedged, only show base assets (exclude hedge instruments for clarity)
    uh_pnl = unhedged['asset_pnl']
    h_pnl  = hedged['asset_pnl'][ASSETS]  # base assets only

    x = np.arange(len(ASSETS))
    width = 0.35
    ax.bar(x - width/2, uh_pnl.values * 100, width, label='Unhedged',
           color=color, alpha=0.5, edgecolor='white')
    ax.bar(x + width/2, h_pnl.values * 100, width, label='Hedged',
           color=color, alpha=0.9, edgecolor='white')
    ax.set_xticks(x)
    ax.set_xticklabels(ASSETS, rotation=30, ha='right', fontsize=9)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_title(f'{name}: Hedge Effectiveness by Asset', fontsize=11)
    ax.legend()
    ax.set_ylabel('P&L (% NAV)')

plt.tight_layout()
plt.show()

---
## Part 3b / Part 6: Crisis Portfolios and Full Comparison

### Crisis Portfolios

A portfolio constructed **knowing** the crisis is coming would deliberately:
- Short or zero out rate-sensitive bonds
- Long oil / energy
- Long defense (Ukraine)
- Long gold and vol
- Reduce EM Asia
- Rotate tourism into quality defensives

In [ ]:
# ── Crisis portfolio weights ──────────────────────────────────────────────────
# These are the full portfolios (base + crisis positioning)
# expressed as combined asset + hedge instrument weights

ALL_ASSETS_EXTENDED = ASSETS + HEDGE_INSTRUMENTS
beta_full = pd.concat([betas, hedge_betas], axis=1)

# Gulf crisis portfolio: positioned for oil spike, inflation, EM stress
w_gulf_crisis = pd.Series({
    '5Y_bonds':       0.10,   # reduced duration
    '30Y_bonds':      0.00,   # zero long duration
    'tourism':        0.00,   # cut completely
    'em_asia_eq':     0.05,   # minimal EM
    'insurance':      0.10,   # reduced
    'sp500_put':      0.05,   # kept
    'cash':           0.15,   # higher cash
    'bond_fut_short': 0.10,   # short duration via futures
    'tips_long':      0.15,   # inflation hedge
    'oil_fut_long':   0.15,   # direct oil exposure
    'usd_call':       0.05,   # EM FX hedge
    'gold_long':      0.10,   # geo risk hedge
    'defense_etf':    0.00,   # not Gulf-specific
})

# Ukraine/Russia crisis portfolio: energy + defense + food + European short
w_ukraine_crisis = pd.Series({
    '5Y_bonds':       0.10,
    '30Y_bonds':      0.00,
    'tourism':        0.05,
    'em_asia_eq':     0.05,
    'insurance':      0.10,
    'sp500_put':      0.05,
    'cash':           0.10,
    'bond_fut_short': 0.10,
    'tips_long':      0.10,
    'oil_fut_long':   0.10,
    'usd_call':       0.05,
    'gold_long':      0.10,
    'defense_etf':    0.10,   # key Ukraine allocation
})

for name, w in [('Gulf crisis portfolio', w_gulf_crisis),
                ('Ukraine crisis portfolio', w_ukraine_crisis)]:
    print(f'{name}: sum = {w.sum():.2f}')

# Stress test crisis portfolios
gulf_crisis_stress    = stress_test(w_gulf_crisis,    beta_full, gulf_shocks)
ukraine_crisis_stress = stress_test(w_ukraine_crisis, beta_full, ukraine_shocks)

In [ ]:
# ── Full comparison: all portfolios under both crises ─────────────────────────
summary = pd.DataFrame({
    'Gulf shock':    [
        gulf_stress['total_pnl'],
        gulf_hedged['total_pnl'],
        gulf_crisis_stress['total_pnl'],
        ukraine_stress['total_pnl'],      # benign under ukraine
    ],
    'Ukraine shock': [
        ukraine_stress['total_pnl'],
        ukraine_hedged['total_pnl'],
        ukraine_stress['total_pnl'],      # placeholder
        ukraine_crisis_stress['total_pnl'],
    ],
}, index=[
    'Benign (unhedged)',
    'Benign + Gulf hedge',
    'Gulf crisis portfolio',
    'Ukraine crisis portfolio',
])

# Re-run properly
summary = pd.DataFrame({
    'Gulf shock': [
        gulf_stress['total_pnl'],
        gulf_hedged['total_pnl'],
        gulf_crisis_stress['total_pnl'],
        stress_test(w_ukraine_crisis, beta_full, gulf_shocks)['total_pnl'],
    ],
    'Ukraine shock': [
        ukraine_stress['total_pnl'],
        ukraine_hedged['total_pnl'],
        stress_test(w_gulf_crisis, beta_full, ukraine_shocks)['total_pnl'],
        ukraine_crisis_stress['total_pnl'],
    ],
}, index=[
    'Benign (unhedged)',
    'Benign + hedge',
    'Gulf crisis portfolio',
    'Ukraine crisis portfolio',
])

print('=== FULL COMPARISON: P&L (% NAV) ===')
print(summary.applymap(lambda x: f'{x:+.2%}').to_string())

# Heatmap
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(
    summary * 100, annot=True, fmt='.1f', cmap='RdYlGn',
    center=0, ax=ax, linewidths=0.5,
    annot_kws={'size': 11, 'weight': 'bold'},
    cbar_kws={'label': 'P&L (% NAV)'}
)
ax.set_title('Portfolio P&L under Each Crisis Scenario (% NAV)', fontsize=12, pad=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Factor exposure comparison across all portfolios ─────────────────────────
portfolios = {
    'Benign':            (w_benign,                          betas),
    'Gulf hedged':       (pd.concat([w_benign, h_gulf]),     beta_full),
    'Ukraine hedged':    (pd.concat([w_benign, h_ukraine]),  beta_full),
    'Gulf crisis ptf':   (w_gulf_crisis,                     beta_full),
    'Ukraine crisis ptf':(w_ukraine_crisis,                  beta_full),
}

exposures = {}
for name, (w, b) in portfolios.items():
    B_p = b @ w
    exposures[name] = B_p

exp_df = pd.DataFrame(exposures)  # shape: (n_factors, n_portfolios)

# Grouped bar chart: factors on x-axis, portfolios as bar groups
n_factors    = len(exp_df)
n_portfolios = len(exp_df.columns)
x      = np.arange(n_factors)
width  = 0.15
colors = ['#1565c0', '#2e7d32', '#b71c1c', '#6a1b9a', '#e65100']

fig, ax = plt.subplots(figsize=(16, 6))
for i, (col, color) in enumerate(zip(exp_df.columns, colors)):
    offset = (i - n_portfolios / 2 + 0.5) * width
    ax.bar(x + offset, exp_df[col].values, width,
           label=col, color=color, alpha=0.85, edgecolor='white')

ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(exp_df.index, rotation=40, ha='right', fontsize=8)
ax.set_title('Factor Exposures Across All Portfolios', fontsize=13)
ax.set_ylabel('Factor Beta')
ax.legend(loc='lower right', fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

---
## Summary and Key Takeaways

### What the factor model revealed

- The benign portfolio had **concentrated unintended factor risk**: 40% in bonds gave a large negative real rates and inflation beta, which is invisible if you only look at asset weights
- The S&P put **did not address the actual risk**: the crisis is a stagflationary factor shock, not a broad equity crash. Hedging the wrong factor is costly and ineffective
- The **insurance leg compounded** the bond book risk rather than diversifying it: insurance equities carry their own duration exposure

### Hedging lessons

| Crisis | Primary hedge | Secondary hedge | What the naive hedge missed |
|---|---|---|---|
| Gulf | Short bond futures + TIPS | Long oil, gold | S&P put useless, index held up |
| Ukraine | Short bond futures + TIPS | Long defense, agriculture | European equity short needed |

### The factor model feedback loop

$$
\text{Build} \xrightarrow{\text{factor exposures}} \text{Stress test} \xrightarrow{\text{attribution}} \text{Hedge} \xrightarrow{\text{retest}} \text{Confirm}
$$

The value is not in the model itself but in the **discipline of decomposing every position into factor language** before a crisis arrives. A geopolitical shock is not unpredictable in factor space: oil, geo risk, real rates, and USD are the transmission channels. The portfolio manager who speaks that language can act faster and hedge more precisely than one thinking in asset terms alone.